在数据预处理阶段，**插值法（Interpolation）** 主要用于**补全缺失值（Missing Data Imputation）**或进行**数据平滑与加密**。

拉格朗日插值和牛顿插值是代数插值的两种核心表现形式。它们的本质相同（对于 $n+1$ 个点，它们得到的最终多项式是唯一的），但其构造逻辑和计算过程大不相同。

---

### 一、 拉格朗日插值法 (Lagrange Interpolation)

#### 1. 算法原理与数学推导
**核心思想**：通过构造一组**基函数（Basis Polynomials）**，将每个节点的函数值“加权”组合起来。

**数学推导**：
假设有 $n+1$ 个离散点 $(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n)$。
我们希望构造一个 $n$ 次多项式 $L(x)$，使得在每个已知点上 $L(x_i) = y_i$。

为此，我们构造基函数 $l_i(x)$，要求它在 $x_i$ 处为 1，在其他 $x_j$（$j \neq i$）处为 0。
基函数的构造公式为：
$$l_i(x) = \prod_{j=0, j \neq i}^{n} \frac{x - x_j}{x_i - x_j}$$
最终的拉格朗日插值多项式为：
$$L(x) = \sum_{i=0}^{n} y_i l_i(x)$$

**数学逻辑**：
当 $x = x_k$ 时，$l_k(x_k) = 1$，而对于所有 $i \neq k$，$l_i(x_k) = 0$。因此 $L(x_k) = y_k \cdot 1 + 0 + \dots = y_k$。

#### 2. 适用场景
*   **缺失值填补**：在时间序列或观测数据中，利用缺失点前后的已知点拟合曲线，预测缺失点的值。
*   **手动计算**：公式结构对称，适合小规模数据的理论推导。
*   **缺点**：结构僵硬，每增加一个新点，所有基函数都得重算；高次插值易出现 **龙格现象（Runge's phenomenon）**，即边缘震荡。

#### 3. Python 代码框架
在数据预处理中，我们常用 `scipy.interpolate` 库，或者手动实现简单的插值函数。

```python
import numpy as np
from scipy.interpolate import lagrange

def lagrange_imputation(x_known, y_known, x_missing):
    """
    拉格朗日插值补全缺失值
    :param x_known: 已知数据的索引(坐标)
    :param y_known: 已知数据的值
    :param x_missing: 缺失数据的索引(待补全点)
    :return: 补全后的值
    """
    # 构造拉格朗日多项式对象
    poly = lagrange(x_known, y_known)

    # 预测缺失值
    # poly.coef 获取多项式系数
    return poly(x_missing)

# --- 示例 ---
x = np.array([1, 2, 3, 5]) # 索引为4的点缺失
y = np.array([10, 15, 12, 18])
x_miss = 4

val = lagrange_imputation(x, y, x_miss)
print(f"拉格朗日插值预测索引 {x_miss} 的值为: {val:.4f}")
```

---

### 二、 牛顿插值法 (Newton Interpolation)

#### 1. 算法原理与数学推导
**核心思想**：利用**差商（Divided Differences）**逐阶构造多项式。它具有**承袭性**，即增加一个点只需要在原有多项式末尾增加一项。

**数学推导**：
定义 $k$ 阶差商为 $f[x_0, x_1, \dots, x_k]$。
*   一阶差商：$f[x_i, x_j] = \frac{y_j - y_i}{x_j - x_i}$
*   二阶差商：$f[x_i, x_j, x_k] = \frac{f[x_j, x_k] - f[x_i, x_j]}{x_k - x_i}$

牛顿插值多项式形式为：
$$P(x) = f(x_0) + f[x_0, x_1](x - x_0) + f[x_0, x_1, x_2](x - x_0)(x - x_1) + \dots$$
即：
$$P(x) = a_0 + a_1(x-x_0) + a_2(x-x_0)(x-x_1) + \dots + a_n \prod_{i=0}^{n-1}(x-x_i)$$
其中 $a_k$ 就是 $k$ 阶差商。

#### 2. 适用场景
*   **动态数据**：如果数据点是不断增加的，牛顿插值只需计算新增的差商项，效率极高。
*   **计算差分表**：在数学建模论文中展示差商表（Divided Difference Table）能显著增加专业度。

#### 3. Python 代码框架
由于 `scipy` 没有直接提供牛顿插值函数（通常用 B-Spline 代替），我们手动实现以便于理解逻辑。

```python
import numpy as np

def newton_interpolation(x_known, y_known, x_predict):
    """
    牛顿插值手动实现
    """
    n = len(x_known)
    # 初始化差商表 (n x n 矩阵)
    coef = np.zeros([n, n])
    coef[:, 0] = y_known

    # 计算差商表
    for j in range(1, n):
        for i in range(n - j):
            coef[i][j] = (coef[i+1][j-1] - coef[i][j-1]) / (x_known[i+j] - x_known[i])

    # 获取差商系数 (矩阵的第一行)
    a = coef[0, :]

    # 构建多项式并计算 x_predict
    res = a[0]
    product = 1.0
    for i in range(1, n):
        product *= (x_predict - x_known[i-1])
        res += a[i] * product
    return res

# --- 示例 ---
x = np.array([1, 2, 3, 5])
y = np.array([10, 15, 12, 18])
print(f"牛顿插值预测值: {newton_interpolation(x, y, 4):.4f}")
```

---

### 三、 论文写作与建模实战建议

在数学建模论文的“数据预处理”章节，你可以这样描述插值法的应用：

#### 1. 建模逻辑表述：
> “由于传感器故障/统计误差，原始数据在 $t_k$ 时刻存在缺失。为保证数据的连续性与后续模型的完整性，本文采用了拉格朗日插值法对缺失值进行修正。通过选取缺失点邻域内的 $n$ 个观测点作为节点构造插值基函数，拟合出局部代数多项式，从而推导出缺失时刻的估计值。”

#### 2. 注意事项（修修补补）：
*   **局部插值 vs 全局插值**：不要用全量数据（比如1000个点）去做一次拉格朗日插值，那样多项式阶数太高，结果会完全失真。
*   **分段线性插值**：实际工程中，为了防止震荡，通常只选取缺失值**前后各 2-3 个点**进行插值（如局部 4 阶插值）。
*   **异常值敏感**：插值法对异常值（Outliers）极其敏感。如果已知点里有一个脏数据，插值出来的缺失值会离谱。因此，**先剔除异常值，再进行插值**。

#### 3. 算法对比：
*   **拉格朗日**：公式简洁，一气呵成，适合静态、离散数据的补全。
*   **牛顿**：逻辑递进，计算过程具有模块化特点，适合需要展示差商推导过程的论文。

**下一类数据预处理算法（如：异常值检测 3σ 原则、PCA 降维、等宽/等频离散化），你想听哪一个？**